# IPL Model Training — Model A

This notebook trains classical ML models using the handcrafted features generated in `IPL_EDA_FeatureEngineering.ipynb`.

Models:
- Random Forest
- SVM + PCA
- XGBoost

Evaluation:
- Accuracy
- Precision (weighted)
- Recall (weighted)
- F1 Score (weighted)

Output:
- Best model selection
- Export as `model_<teamname>.pkl`


## 1. Import Libraries

In [3]:
# --- progress helpers (added) ---------------------------------------------
import time as _time
from contextlib import contextmanager
@contextmanager
def step(msg):
    print(f'\u23f3 {msg} ...', end='', flush=True); _t0=_time.time()
    try:
        yield
    finally:
        print(f'\r\u2705 {msg} \u2014 done in {_time.time()-_t0:.1f}s'+' '*8, flush=True)
# --------------------------------------------------------------------------


import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier

import joblib
from pathlib import Path


## 2. Load Features Dataset

In [5]:

DATA_DIR = Path("../data")

FEATURE_FILE = DATA_DIR / "features.csv"

df = pd.read_csv(FEATURE_FILE)

print("Shape:", df.shape)
df.head()


Shape: (230720, 679)


,image_name,cell_id,label,f_1,f_2,f_3,f_4,f_5,f_6,f_7,...,f_667,f_668,f_669,f_670,f_671,f_672,f_673,f_674,f_675,f_676
0,GTvsLSG_image_0.jpg,1,0,0.819521,0.273765,0.267433,0.419384,0.057488,0.051157,0.010637,...,0.037274,0.029713,0.090555,0.213707,0.275448,0.275448,0.141855,0.093850,0.041015,0.020408
1,GTvsLSG_image_0.jpg,2,0,0.007270,0.100509,0.512661,0.542688,0.515190,0.195962,0.195330,...,0.055488,0.307377,0.084778,0.045963,0.056343,0.064856,0.095264,0.105586,0.182372,0.307377
2,GTvsLSG_image_0.jpg,3,0,0.000000,0.042769,0.142231,0.323585,0.794706,0.330879,0.268880,...,0.066395,0.046282,0.010631,0.023346,0.081757,0.291008,0.240174,0.291008,0.291008,0.125004
3,GTvsLSG_image_0.jpg,4,0,0.093658,0.460096,0.391803,0.332096,0.277462,0.178731,0.353169,...,0.035454,0.080711,0.136827,0.293666,0.293666,0.202032,0.078363,0.044881,0.019489,0.006625
4,GTvsLSG_image_0.jpg,5,0,0.243183,0.841610,0.473506,0.079453,0.028934,0.025030,0.022504,...,0.074625,0.230121,0.166748,0.114429,0.033296,0.089340,0.106698,0.107236,0.129191,0.106516


## 3. Prepare Features and Labels

In [7]:

feature_cols = [c for c in df.columns if c.startswith("f_")]

X = df[feature_cols]
y = df["label"]
groups = df["image_name"]

# clean non-finite values (NaN/inf) now that X exists
bad = (~np.isfinite(X)).to_numpy().sum()
print("Non-finite feature cells:", int(bad))
X = np.nan_to_num(X.to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)

print("Features:", X.shape)
print("Labels:", y.shape)
print("Images:", groups.nunique())


Non-finite feature cells: 0
Features: (230720, 676)
Labels: (230720,)
Images: 3605


## 4. Image-Aware Train/Test Split (GroupShuffleSplit)

In [9]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X[train_idx]      # array indexing, not .iloc
X_test  = X[test_idx]

y_train = y.iloc[train_idx]  # y is still a Series -> .iloc is correct
y_test  = y.iloc[test_idx]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (184576, 676)
Test : (46144, 676)


## 5. Random Forest

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

with step('Training Random Forest'):
    rf_model.fit(X_train, y_train)
with step('Random Forest: predicting test set'):
    rf_pred = rf_model.predict(X_test)


✅ Training Random Forest — done in 240.8s        
✅ Random Forest: predicting test set — done in 0.7s        


## 6. SVM with PCA

In [ ]:

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("svm", SVC(
        kernel="rbf",
        C=10,
        gamma="scale"
    ))
])

with step('Training SVM+PCA (RBF SVM is slow on large data \u2014 be patient)'):
    svm_pipeline.fit(X_train, y_train)
with step('SVM+PCA: predicting test set'):
    svm_pred = svm_pipeline.predict(X_test)


⏳ Training SVM+PCA (RBF SVM is slow on large data — be patient) ...

## 7. XGBoost

In [11]:

num_classes = len(np.unique(y))

xgb_model = XGBClassifier(
    objective="multi:softmax",
    num_class=num_classes,
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

with step('Training XGBoost'):
    xgb_model.fit(X_train, y_train)
with step('XGBoost: predicting test set'):
    xgb_pred = xgb_model.predict(X_test)


✅ Training XGBoost — done in 1724.4s        
✅ XGBoost: predicting test set — done in 1.9s        


## 8. Evaluation Function

In [ ]:

def evaluate_model(name, y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    rec = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print(f"\n{name}")
    print("-"*50)
    print("Accuracy :", round(acc,4))
    print("Precision:", round(prec,4))
    print("Recall   :", round(rec,4))
    print("F1 Score :", round(f1,4))

    return {
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    }


## 9. Compare Models

In [ ]:

results = []

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred
    )
)

results.append(
    evaluate_model(
        "SVM + PCA",
        y_test,
        svm_pred
    )
)

results.append(
    evaluate_model(
        "XGBoost",
        y_test,
        xgb_pred
    )
)

results_df = pd.DataFrame(results)
results_df.sort_values(
    by="F1",
    ascending=False
)


## 10. Select Best Model

In [ ]:

best_row = results_df.sort_values(
    by="F1",
    ascending=False
).iloc[0]

best_model_name = best_row["Model"]

if best_model_name == "Random Forest":
    best_model = rf_model
elif best_model_name == "SVM + PCA":
    best_model = svm_pipeline
else:
    best_model = xgb_model

print("Best Model:", best_model_name)
print(best_row)


## 11. Save Best Model

In [ ]:

TEAM_NAME = "teamname"   # Replace with your actual team name

model_file = f"model_{TEAM_NAME}.pkl"

joblib.dump(best_model, model_file)

print("Saved:", model_file)


## 12. Detailed Classification Report

In [ ]:

with step('Best model: predicting test set for final report'):
    best_predictions = best_model.predict(X_test)

print(
    classification_report(
        y_test,
        best_predictions,
        zero_division=0
    )
)
